# PhonAI Bot


## Libraries

### Libraries install

In [18]:
import sys
# !{sys.executable} -m pip install --upgrade youtube-transcript-api langchain-text-splitters datasets numpy langchain langchain_community langchain_openai langchain-pinecone pinecone-client gradio

> Restart Kernel after

### Libraries import

In [27]:
import sys
import os
import time
import gradio as gr
from tqdm.auto import tqdm
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# YouTube & Text Splitting
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Pinecone & Embeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Modern LangChain Core & LangGraph Agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import create_retriever_tool
from langgraph.prebuilt import create_react_agent

## 0. Environment Setup & LangSmith Tracing

In [20]:
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# LangSmith Integration for Tracing & Evaluation
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "PhoneAI_Project")

# Initialize standard OpenAI client for Whisper API
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Constants
INDEX_NAME = 'phoneai-bot'

## 1. Youtube Transcript API
Get videos transcriptions

In [ ]:
video_ids = [
    'ZdO82BBni0M', # SuperSaf - Which SMARTPHONES Do We Actually Use? 2026 Edition ft. MKBHD, Linus Tech Tips + More
    'HGYBnOk5RSk', # SuperSaf - Top 5 BEST Smartphones of 2026… So Far
    'CPWK60ch7BM', # Monsieur GRrr - WHICH SMARTPHONE TO BUY in SPRING 2026?
    'SDz_3OTDTqI', # Romain PIAUD - I finally understood which SMARTPHONE to choose in 2026... (and that changes everything)
    'YNZW6L13IYs', # MobileTechReview -Samsung Galaxy S26 Ultra Review 
    'd8HlcRljhMY', # Nooz - The best mid-range smartphones (2026) – my #1 choice will divide
    'pULPu5_re5Q', # Monde Tech - TOP-5: The best mid-range smartphones – Test 2026!
    '2MVB-YleUCk', # Frandroid - The 3 Samsung GALAXY S26: Our GETTING STARTED and all the NEW FEATURES
    'xRla0izRxqU', # Mrwhosetheboss - iPhone 17 Pro Review - Something’s Missing
    'Y9JF_yYTNlw', # Mrwhosetheboss - The BEST Smartphones of 2025! 
    '-gO13Xt4TSQ', # GSMArena Official - Samsung Galaxy S26 Ultra vs iPhone 17 Pro Max: Which one to get?
    '7mCTaTJ0kDo', # Mrwhosetheboss - Cheap vs Midrange vs Expensive Phone - Should you spend more?
    'fUddf5iOBFo', # TOUKIWANTI - You are buying the wrong smartphones in 2026
    'mRNvVRPX-uw', # Yensing, Monsieur Rapport Qualité-Prix - Which SMARTPHONE To Buy in 2026? Top 15 Best Value for Money
    'goxV98myylA', # iBordelais - TOP 5 Mid-range Smartphone 2026 – Which one to buy at good value for money?
    'aOkBxWJHnvo', # ASBYT - Best smartphones 2026: The REAL winners (for all budgets)
    '03BUfrW79qw', # DealSmart FR - The best smartphones 2026 – quality that is definitely worth it
    'U8bB8U8j-Gc', # DealSmart FR - The best smartphones 2026 – recommendations for all budgets
    'sfyL4BswUeE', # Marques Brownlee - Smartphone Awards 2025!
    'tYhis0saGnM', # GIGATOP - What is the BEST smartphone for €500 in 2026? (Buying Guide)
    'GrsyYVVXnrI', # BFM Tech - What are the best smartphones at the moment?
    '5dfFraHM2aQ', # merveltech - What budget do you have? Here are the BEST smartphones to buy in 2025-2026
    '1o2NiZmgvus', # Versus - The Ultimate Flagship Comparison: S26 Ultra vs 17 Pro Max vs Pixel 10 Pro XL!
    'I7Nzes9gXc4', # TechWiser - Don't buy smartphones yet! *Next smartphones*
    'rErr-ipjstA', # mobiscrub - One budget = one perfect phone. Your quest for 2026 is over.
    'Zw09YAcMS30', # Techmo - Galaxy S26 vs. iPhone 17, Xiaomi 17, Pixel 10: Camera, battery, performance!
    'CjuO8ck2MuM', # Tech Spurt - Best phones of 2025 | I tested more than 75 smartphones, here are my favorites!
    'FKEbkTSoePM', # Tech Spurt - Best Mid-Range Phones (Spring 2026) | Top 20 Reviewed
    'f0QZqdEbIL4', # Versus -  Best Smartphone for Vlogging in 2026: Not What You’d Expect! 
]
ytt_api = YouTubeTranscriptApi()

# 1. Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)

all_chunks = []
all_metadatas = []

print("Starting Transcript Extraction...")
for video_id in video_ids:
    try:
        # Fetch subtitles
        fetched_transcript = ytt_api.fetch(video_id, languages=['fr', 'en'])
        
        full_text = ""
        for snippet in fetched_transcript:
            full_text += snippet.text + " "
        
        # Split the text into chunks
        video_chunks = text_splitter.split_text(full_text)
        
        # Create metadata for each chunk and save them
        for chunk in video_chunks:
            all_chunks.append(chunk)
            all_metadatas.append({
                "text": chunk,
                "video_id": video_id
            })
            
        print(f"Successfully processed {len(video_chunks)} chunks for video {video_id}")
        
        time.sleep(2)
        
    except Exception as e:
        print(f"Error with video {video_id}: {e}")

print(f"Total chunks created for all videos: {len(all_chunks)}")

Starting Transcript Extraction...
Successfully processed 16 chunks for video ZdO82BBni0M
Successfully processed 20 chunks for video HGYBnOk5RSk
Successfully processed 14 chunks for video CPWK60ch7BM
Successfully processed 26 chunks for video SDz_3OTDTqI
Successfully processed 18 chunks for video YNZW6L13IYs
Successfully processed 16 chunks for video d8HlcRljhMY
Successfully processed 11 chunks for video pULPu5_re5Q
Successfully processed 19 chunks for video 2MVB-YleUCk
Successfully processed 16 chunks for video xRla0izRxqU
Successfully processed 27 chunks for video Y9JF_yYTNlw
Successfully processed 12 chunks for video -gO13Xt4TSQ
Successfully processed 25 chunks for video 7mCTaTJ0kDo
Successfully processed 14 chunks for video fUddf5iOBFo
Successfully processed 23 chunks for video mRNvVRPX-uw
Successfully processed 33 chunks for video goxV98myylA
Successfully processed 18 chunks for video aOkBxWJHnvo
Successfully processed 16 chunks for video 03BUfrW79qw
Successfully processed 19 chunk

## 2. Pinecone Index Setup & Upsert
### Index Setup

In [22]:
# 1. Initialize the Embeddings model
embed = OpenAIEmbeddings(
    model='text-embedding-ada-002',
    openai_api_key=OPENAI_API_KEY
)

# 2. Connect to Pinecone and create index if needed
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating new index: {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # Ada-002 dimension
        metric='dotproduct', 
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(1)

# 3. Connect to the index
index = pc.Index(INDEX_NAME)
print("Pinecone index successfully created!")
print("Index stats:", index.describe_index_stats())

Pinecone index successfully created!
Index stats: {'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 552}},
 'total_vector_count': 552,
 'vector_type': 'dense'}


### Pinecone Upsert

In [23]:
if index.describe_index_stats().get("total_vector_count", 0) == 0:
    batch_size = 100

    # We iterate through all the chunks we created
    for i in tqdm(range(0, len(all_chunks), batch_size)):
        i_end = min(len(all_chunks), i + batch_size)
        
        # 1. Retrieve the text documents and metadata for this batch
        documents_batch = all_chunks[i:i_end]
        metadatas_batch = all_metadatas[i:i_end]
        
        # 2. Generate embeddings using OpenAI
        embeds = embed.embed_documents(documents_batch)
        
        # 3. Generate unique IDs for each vector 
        ids = [f"chunk_{j}" for j in range(i, i_end)]
        
        # 4. Upsert into Pinecone
        index.upsert(vectors=zip(ids, embeds, metadatas_batch))
        
    print("Pinecone index successfully populated!")
    print("Index stats:", index.describe_index_stats())

## ➡️ 3. Create LangChain RAG & Agent

In [24]:
# 1. Connect to Pinecone vector database
text_field = "text"  
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME, 
    embedding=embed, 
    text_key=text_field
)

# 2. Create the Retriever Tool
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
tool_description = (
    "Use this tool to search for smartphone specs, reviews, and tech advice. "
    "Always use this tool when the user asks about smartphones, phone recommendations, or tech. "
    "Do NOT use this tool for general greetings, math calculations, or unrelated topics."
)
retriever_tool = create_retriever_tool(
    retriever,
    "smartphone_tech_database",
    tool_description
)
tools = [retriever_tool]

# 3. Initialize the LLM with STREAMING enabled
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-4o-mini', 
    temperature=0.0,
    streaming=True
)

# 4. Advanced System Prompt
phone_ai_system_prompt = """You are 'PhoneAI', an elite and highly technical smartphone advisor.
Your primary directive is to guide users to the best smartphone choices.

CRITICAL INSTRUCTIONS YOU MUST FOLLOW:
1. Tool Usage: If the user asks about smartphones, specs, reviews, or recommendations, you MUST use the smartphone_tech_database tool to find answers.
2. Out-of-Domain Knowledge: If the user asks a general question (e.g., "Hello", math problems, history), do NOT use the tool. Answer directly using your internal knowledge.
3. Cross-Lingual Synthesis: Your knowledge base contains data in multiple languages. You must synthesize this data into the user's requested language.
4. Natural Conversational Tone: You are strictly forbidden from using robotic phrases like "Based on the tool" or "According to the database". Answer directly.
5. Honesty Limit: If the tool does not contain the answer to a smartphone question, simply state that you do not have the information. Do not hallucinate.
"""
system_message = SystemMessage(content=phone_ai_system_prompt)

# 5. Construct the Modern LangGraph Agent
# In LangGraph v1.1+, the parameter was consolidated to simply 'prompt'
agent_executor = create_react_agent(
    model=llm, 
    tools=tools, 
    prompt=system_message 
)

C:\Users\Phiph\AppData\Local\Temp\ipykernel_14540\2924293085.py:46: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


## 4. Gradio deployment
Gradio Multimodal UI (Whisper + Chat)
### Streaming message

In [ ]:
# ==========================================
# 4. Gradio Multimodal UI (Whisper + LangGraph Streaming + TTS)
# ==========================================

def process_interaction_streaming(audio_filepath, text_input, chat_history):
    """
    Handles voice/text inputs, streams the Agent's text response back to the UI in real-time,
    and plays the final response as audio using OpenAI's TTS.
    """
    user_message = ""
    
    # 1. Process Audio Input with Whisper
    if audio_filepath:
        try:
            with open(audio_filepath, "rb") as audio_file:
                transcript = openai_client.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file
                )
            user_message = transcript.text
        except Exception as e:
            user_message = f"[Error processing audio]: {str(e)}"
            
    # 2. Process Text Input (if no audio)
    elif text_input:
        user_message = text_input

    # 3. Skip if empty
    if not user_message.strip():
        # Notice the 4th yielded value (None) corresponds to the new audio_output component
        yield None, "", chat_history, None
        return

    # 4. Instantly update UI with user message and create an empty assistant message slot
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": ""})
    yield None, "", chat_history, None 

    # 5. Format Gradio History for LangGraph
    formatted_history = []
    for msg in chat_history[:-2]: 
        if msg["role"] == "user":
            formatted_history.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            formatted_history.append(AIMessage(content=msg["content"]))
            
    # Add the current user query
    formatted_history.append(HumanMessage(content=user_message))
    
    # Keep last 10 messages for memory management
    formatted_history = formatted_history[-10:]

    # Variable to capture the complete text for TTS
    final_assistant_text = ""

    # 6. Execute LangGraph Agent and Stream Tokens
    try:
        # stream_mode="messages" allows us to intercept token-by-token chunks safely
        for msg_chunk, metadata in agent_executor.stream(
            {"messages": formatted_history}, 
            stream_mode="messages"
        ):
            # Isolate the agent's generation node to avoid streaming raw tool-call JSON to the UI
            if metadata.get("langgraph_node") == "agent":
                if msg_chunk.content and isinstance(msg_chunk.content, str):
                    chat_history[-1]["content"] += msg_chunk.content
                    final_assistant_text += msg_chunk.content
                    # Yield None for audio while text is still streaming
                    yield None, "", chat_history, None 
                    
        # 7. Generate Text-to-Speech (TTS) from the final generated text
        if final_assistant_text.strip():
            audio_output_path = "response.mp3"
            tts_response = openai_client.audio.speech.create(
                model="tts-1",
                voice="alloy",
                input=final_assistant_text
            )
            
            # Save the generated speech to a local file
            tts_response.stream_to_file(audio_output_path)
            
            # Final yield: Update UI with the path to the audio file so it autoplays
            yield None, "", chat_history, audio_output_path
                
    except Exception as e:
        chat_history[-1]["content"] += f"\nAn error occurred: {str(e)}"
        yield None, "", chat_history, None

### UI Block

In [26]:
import gradio as gr

# Custom CSS for a centered layout, theme-adaptive styling, and a clean pill input
custom_css = """
/* Restrict max width and center the entire Gradio app */
.gradio-container {
    max-width: 850px !important;
    margin: 0 auto !important;
}

/* Style the unified input bar to look like a single rounded pill */
/* Using CSS variables to adapt to both Light and Dark mode seamlessly */
#unified_input_group {
    border-radius: 30px !important;
    border: 1px solid var(--border-color-primary) !important;
    background-color: var(--background-fill-primary) !important;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05) !important;
    overflow: hidden !important;
    padding: 2px 10px !important;
    align-items: center !important;
    display: flex !important;
}

/* Remove standard borders and backgrounds from the text input inside the group */
#unified_text textarea {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
    font-size: 16px !important;
}

/* Make the audio waveform/button background transparent to blend in */
#unified_audio {
    border: none !important;
    background: transparent !important;
    box-shadow: none !important;
}

/* Add polish to the chatbot bubbles */
.gradio-chatbot {
    border-radius: 16px !important;
    border: 1px solid var(--border-color-primary) !important;
}
"""

modern_theme = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"]
)

with gr.Blocks(title="📱 PhoneAI - Expert Advisor", theme=modern_theme, css=custom_css) as app:
    
    # Minimalistic Header
    gr.Markdown("<h1 style='text-align: center; margin-bottom: 0.2em; font-weight: 700;'>📱 PhoneAI</h1>")
    gr.Markdown("<p style='text-align: center; color: var(--body-text-color-subdued); font-size: 1.1em; margin-top: 0;'>Your Premium Smartphone Expert (2026). Type or speak your query below.</p>")
    
    # Chatbot Interface (bubble_full_width removed for compatibility)
    chatbot_ui = gr.Chatbot(
        show_label=False, 
        height=550, 
        elem_id="main_chat"
    )
    
    # Unified Input Area (Text + Audio together)
    with gr.Group(elem_id="unified_input_group"):
        with gr.Row(equal_height=True):
            txt_input = gr.Textbox(
                show_label=False, 
                placeholder="Message PhoneAI...", 
                container=False, 
                scale=8,
                elem_id="unified_text",
                lines=1,
                max_lines=5
            )
            audio_input = gr.Audio(
                sources=["microphone"], 
                type="filepath", 
                show_label=False,
                container=False, 
                scale=1,
                elem_id="unified_audio"
            )
            
    # Action Buttons placed cleanly below the input
    with gr.Row():
        submit_btn = gr.Button("Submit", variant="primary", scale=4)
        clear_btn = gr.ClearButton(components=[txt_input, audio_input, chatbot_ui], scale=1)

    # Restored Audio Output for Text-to-Speech
    # Placed discreetly below the main controls
    with gr.Accordion("🔊 Text-to-Speech Output", open=False):
        audio_output = gr.Audio(
            label="PhoneAI Voice Response", 
            autoplay=True, 
            visible=True
        )
    
    clear_btn.add(audio_output)

    # Outputs mapping array matching the backend yield structure
    interaction_outputs = [audio_input, txt_input, chatbot_ui, audio_output]

    # Connect events
    submit_btn.click(
        fn=process_interaction_streaming,
        inputs=[audio_input, txt_input, chatbot_ui],
        outputs=interaction_outputs
    )
    
    txt_input.submit(
        fn=process_interaction_streaming,
        inputs=[audio_input, txt_input, chatbot_ui],
        outputs=interaction_outputs
    )
    
    # Changed from 'change' to 'stop_recording' to prevent accidental form submissions while typing
    audio_input.stop_recording(
        fn=process_interaction_streaming,
        inputs=[audio_input, txt_input, chatbot_ui],
        outputs=interaction_outputs
    )

if __name__ == "__main__":
    app.launch(share=True)

C:\Users\Phiph\AppData\Local\Temp\ipykernel_14540\919931207.py:52: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="📱 PhoneAI - Expert Advisor", theme=modern_theme, css=custom_css) as app:


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://95b4dcaca4902d06f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
